In [1]:
import nba_api.stats.endpoints
import requests
import json
import pandas as pd
import nba_api

In [2]:
# Get Timberwolves player game logs for the season
wolves_player_logs = nba_api.stats.endpoints.PlayerGameLogs(season_nullable='2025-26',team_id_nullable='1610612750').get_data_frames()[0]

wolves_player_logs

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,...,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT
0,2025-26,1630162,Anthony Edwards,Anthony,1610612750,MIN,Minnesota Timberwolves,0022500714,2026-02-02T00:00:00,MIN @ MEM,...,2,8,281,15,46,3,12,1,39:41,1
1,2025-26,1630183,Jaden McDaniels,Jaden,1610612750,MIN,Minnesota Timberwolves,0022500714,2026-02-02T00:00:00,MIN @ MEM,...,119,34,281,102,46,3,74,1,36:50,1
2,2025-26,203944,Julius Randle,Julius,1610612750,MIN,Minnesota Timberwolves,0022500714,2026-02-02T00:00:00,MIN @ MEM,...,46,104,217,86,46,3,90,1,40:15,1
3,2025-26,1628978,Donte DiVincenzo,Donte,1610612750,MIN,Minnesota Timberwolves,0022500714,2026-02-02T00:00:00,MIN @ MEM,...,189,83,420,132,46,3,99,1,33:40,1
4,2025-26,1629675,Naz Reid,Naz,1610612750,MIN,Minnesota Timberwolves,0022500714,2026-02-02T00:00:00,MIN @ MEM,...,295,273,631,175,46,3,206,1,25:49,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
643,2025-26,204060,Joe Ingles,Joe,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,442,462,281,486,46,3,467,1,16:10,1
644,2025-26,1642389,Zyon Pullin,Zyon,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,442,499,404,563,46,3,545,1,10:19,1
645,2025-26,1631262,Jules Bernard,Jules,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,442,558,420,576,46,3,581,1,4:10,1
646,2025-26,1641803,Tristen Newton,Tristen,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,442,499,281,592,46,3,581,1,3:08,1


In [3]:
# Filter for Anthony Edwards and Bones Hyland game logs
edwards_logs = wolves_player_logs[wolves_player_logs['PLAYER_NAME'] == 'Anthony Edwards'].copy()
hyland_logs = wolves_player_logs[wolves_player_logs['PLAYER_NAME'] == 'Bones Hyland'].copy()

print(f"Anthony Edwards games: {len(edwards_logs)}")
print(f"Bones Hyland games: {len(hyland_logs)}")

# Get unique Game IDs for each player
edwards_game_ids = set(edwards_logs['GAME_ID'].unique())
hyland_game_ids = set(hyland_logs['GAME_ID'].unique())

print(f"\nEdwards Game IDs: {len(edwards_game_ids)}")
print(f"Hyland Game IDs: {len(hyland_game_ids)}")

# Find games where Hyland played but Edwards did NOT play
hyland_without_edwards_game_ids = hyland_game_ids - edwards_game_ids

print(f"\nGames where Bones Hyland played WITHOUT Anthony Edwards: {len(hyland_without_edwards_game_ids)}")

Anthony Edwards games: 44
Bones Hyland games: 48

Edwards Game IDs: 44
Hyland Game IDs: 48

Games where Bones Hyland played WITHOUT Anthony Edwards: 13


In [4]:
# Filter Hyland's logs for games where Edwards did NOT play
hyland_without_edwards = hyland_logs[hyland_logs['GAME_ID'].isin(hyland_without_edwards_game_ids)].copy()

print(f"Bones Hyland game logs when Anthony Edwards did NOT play: {len(hyland_without_edwards)}")

if len(hyland_without_edwards) > 0:
    # Sort by date
    hyland_without_edwards = hyland_without_edwards.sort_values('GAME_DATE', ascending=False)
    
    # Display the game logs
    display_cols = ['GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 
                   'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 
                   'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']
    display_cols = [col for col in display_cols if col in hyland_without_edwards.columns]
    
    hyland_without_edwards[display_cols]
else:
    print("No games found where Bones Hyland played without Anthony Edwards")

Bones Hyland game logs when Anthony Edwards did NOT play: 13


In [5]:
# Calculate averages for Bones Hyland when Anthony Edwards does NOT play
if len(hyland_without_edwards) > 0:
    # Select numeric columns for averaging
    numeric_cols = ['MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
                   'FTM', 'FTA', 'FT_PCT', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']
    
    # Filter to only columns that exist in the dataframe
    numeric_cols = [col for col in numeric_cols if col in hyland_without_edwards.columns]
    
    # Calculate averages
    averages = hyland_without_edwards[numeric_cols].mean()
    
    print("Bones Hyland Averages when Anthony Edwards does NOT play:")
    print("=" * 60)
    
    # Format and display averages
    for col in numeric_cols:
        if col in averages.index:
            value = averages[col]
            if 'PCT' in col:
                print(f"{col:15s}: {value:.3f} ({value*100:.1f}%)")
            else:
                print(f"{col:15s}: {value:.2f}")
    
    print("\n" + "=" * 60)
    print(f"Total games: {len(hyland_without_edwards)}")
    
    # Calculate win-loss record
    if 'WL' in hyland_without_edwards.columns:
        wins = (hyland_without_edwards['WL'] == 'W').sum()
        losses = (hyland_without_edwards['WL'] == 'L').sum()
        if wins + losses > 0:
            win_pct = wins / (wins + losses)
            print(f"Team Record: {wins}-{losses} ({win_pct:.3f} / {win_pct*100:.1f}%)")
else:
    print("No games found to calculate averages")

Bones Hyland Averages when Anthony Edwards does NOT play:
MIN            : 18.08
PTS            : 9.08
FGM            : 3.08
FGA            : 6.92
FG_PCT         : 0.430 (43.0%)
FG3M           : 1.62
FG3A           : 4.15
FG3_PCT        : 0.401 (40.1%)
FTM            : 1.31
FTA            : 1.85
FT_PCT         : 0.279 (27.9%)
REB            : 2.54
AST            : 3.38
STL            : 0.54
BLK            : 0.15
TOV            : 1.23
PF             : 2.54
PLUS_MINUS     : 6.69

Total games: 13
Team Record: 8-5 (0.615 / 61.5%)


In [6]:
# Compare: Hyland's averages WITH vs WITHOUT Edwards
print("Comparison: Bones Hyland WITH vs WITHOUT Anthony Edwards")
print("=" * 70)

# Calculate Hyland's averages in ALL games (including with Edwards)
hyland_all_games = hyland_logs.copy()
hyland_with_edwards = hyland_logs[hyland_logs['GAME_ID'].isin(edwards_game_ids)].copy()

if len(hyland_without_edwards) > 0 and len(hyland_with_edwards) > 0:
    numeric_cols = ['MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
                   'FTM', 'FTA', 'FT_PCT', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']
    numeric_cols = [col for col in numeric_cols if col in hyland_logs.columns]
    
    avg_without = hyland_without_edwards[numeric_cols].mean()
    avg_with = hyland_with_edwards[numeric_cols].mean()
    avg_all = hyland_all_games[numeric_cols].mean()
    
    print(f"\n{'Stat':<15} {'WITHOUT Edwards':<20} {'WITH Edwards':<20} {'ALL Games':<20}")
    print("-" * 70)
    
    for col in numeric_cols:
        if col in avg_without.index and col in avg_with.index:
            without_val = avg_without[col]
            with_val = avg_with[col]
            all_val = avg_all[col]
            
            if 'PCT' in col:
                print(f"{col:<15} {without_val:>6.3f} ({without_val*100:>5.1f}%){'':<6} {with_val:>6.3f} ({with_val*100:>5.1f}%){'':<6} {all_val:>6.3f} ({all_val*100:>5.1f}%)")
            else:
                print(f"{col:<15} {without_val:>6.2f}{'':<13} {with_val:>6.2f}{'':<13} {all_val:>6.2f}")
    
    print("\n" + "=" * 70)
    print(f"Games WITHOUT Edwards: {len(hyland_without_edwards)}")
    print(f"Games WITH Edwards: {len(hyland_with_edwards)}")
    print(f"Total games: {len(hyland_all_games)}")
else:
    print(f"\nGames WITHOUT Edwards: {len(hyland_without_edwards)}")
    print(f"Games WITH Edwards: {len(hyland_with_edwards)}")
    if len(hyland_without_edwards) == 0:
        print("\nNo games found where Bones Hyland played without Anthony Edwards")

Comparison: Bones Hyland WITH vs WITHOUT Anthony Edwards

Stat            WITHOUT Edwards      WITH Edwards         ALL Games           
----------------------------------------------------------------------
MIN              18.08               13.10               14.45
PTS               9.08                5.89                6.75
FGM               3.08                2.14                2.40
FGA               6.92                4.89                5.44
FG_PCT           0.430 ( 43.0%)        0.395 ( 39.5%)        0.404 ( 40.4%)
FG3M              1.62                1.09                1.23
FG3A              4.15                3.00                3.31
FG3_PCT          0.401 ( 40.1%)        0.311 ( 31.1%)        0.336 ( 33.6%)
FTM               1.31                0.51                0.73
FTA               1.85                0.71                1.02
FT_PCT           0.279 ( 27.9%)        0.290 ( 29.0%)        0.287 ( 28.7%)
REB               2.54                1.40                1.

In [7]:
# Display Bones Hyland's complete game logs when Anthony Edwards does NOT play
from IPython.display import display

print("Bones Hyland Game Logs (Games where Anthony Edwards did NOT play):")
print("=" * 100)

if len(hyland_without_edwards) > 0:
    # Ensure sorted by date (most recent first)
    hyland_without_edwards_display = hyland_without_edwards.sort_values('GAME_DATE', ascending=False).copy()
    
    # Select columns to display
    display_cols = ['GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 
                   'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 
                   'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']
    display_cols = [col for col in display_cols if col in hyland_without_edwards_display.columns]
    
    # Display the dataframe
    display(hyland_without_edwards_display[display_cols])
else:
    print("No games found where Bones Hyland played without Anthony Edwards")

Bones Hyland Game Logs (Games where Anthony Edwards did NOT play):


,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,...,FTM,FTA,FT_PCT,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS
44,2026-01-26T00:00:00,MIN vs. GSW,W,24.733333,17,6,9,0.667,3,4,...,2,4,0.500,7,5,2,0,1,4,19
104,2026-01-16T00:00:00,MIN @ HOU,L,16.901667,2,1,4,0.250,0,2,...,0,2,0.000,0,5,0,0,1,1,-11
108,2026-01-13T00:00:00,MIN @ MIL,W,21.050000,23,9,16,0.563,5,7,...,0,0,0.000,2,5,1,0,0,2,41
269,2025-12-17T00:00:00,MIN vs. MEM,L,34.138333,12,3,11,0.273,2,9,...,4,5,0.800,3,5,1,0,2,5,10
277,2025-12-14T00:00:00,MIN vs. SAC,W,35.683333,18,6,10,0.600,1,4,...,5,5,1.000,3,5,0,1,2,5,10
290,2025-12-12T00:00:00,MIN @ GSW,W,4.650000,3,1,1,1.000,1,1,...,0,0,0.000,0,2,0,0,0,1,2
489,2025-11-03T00:00:00,MIN @ BKN,W,5.183333,0,0,2,0.000,0,2,...,0,0,0.000,1,1,1,0,0,0,-5
500,2025-11-01T00:00:00,MIN @ CHA,W,9.966667,3,1,3,0.333,1,3,...,0,0,0.000,1,1,0,0,0,2,-2
511,2025-10-29T00:00:00,MIN vs. LAL,L,8.376667,8,3,4,0.750,2,3,...,0,0,0.000,1,2,0,0,2,3,-1
521,2025-10-27T00:00:00,MIN vs. DEN,L,15.616667,4,1,6,0.167,1,5,...,1,2,0.500,2,1,0,0,2,2,-4


In [9]:
# Calculate Timberwolves record when Bones Hyland scores 10+ points
from IPython.display import display

# Filter for games where Bones Hyland scored 10 or more points
hyland_10plus_pts = hyland_logs[hyland_logs['PTS'] >= 12].copy()

print("Timberwolves Record when Bones Hyland scores 10+ points:")
print("=" * 70)

if len(hyland_10plus_pts) > 0:
    wins = (hyland_10plus_pts['WL'] == 'W').sum()
    losses = (hyland_10plus_pts['WL'] == 'L').sum()
    total = wins + losses
    
    if total > 0:
        win_pct = wins / total
        print(f"\nGames where Bones Hyland scored 10+ points: {total}")
        print(f"Wins: {wins}")
        print(f"Losses: {losses}")
        print(f"Win Percentage: {win_pct:.3f} ({win_pct*100:.1f}%)")
        
        # Show game details
        print(f"\nGame Details:")
        display_cols = ['GAME_DATE', 'MATCHUP', 'WL', 'PTS', 'MIN', 'FGM', 'FGA', 'FG_PCT', 
                       'FG3M', 'FG3A', 'AST', 'REB', 'PLUS_MINUS']
        display_cols = [col for col in display_cols if col in hyland_10plus_pts.columns]
        
        hyland_10plus_display = hyland_10plus_pts[display_cols].sort_values('GAME_DATE', ascending=False)
        display(hyland_10plus_display)
        
        # Compare to overall record
        print(f"\nComparison:")
        print(f"Total games played by Bones Hyland: {len(hyland_logs)}")
        overall_wins = (hyland_logs['WL'] == 'W').sum()
        overall_losses = (hyland_logs['WL'] == 'L').sum()
        overall_total = overall_wins + overall_losses
        if overall_total > 0:
            overall_win_pct = overall_wins / overall_total
            print(f"Overall record when Bones Hyland plays: {overall_wins}-{overall_losses} ({overall_win_pct:.3f} / {overall_win_pct*100:.1f}%)")
    else:
        print("Could not determine win/loss record")
else:
    print("No games found where Bones Hyland scored 10+ points")

Timberwolves Record when Bones Hyland scores 10+ points:

Games where Bones Hyland scored 10+ points: 10
Wins: 7
Losses: 3
Win Percentage: 0.700 (70.0%)

Game Details:


,GAME_DATE,MATCHUP,WL,PTS,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,AST,REB,PLUS_MINUS
44,2026-01-26T00:00:00,MIN vs. GSW,W,17,24.733333,6,9,0.667,3,4,5,7,19
108,2026-01-13T00:00:00,MIN @ MIL,W,23,21.050000,9,16,0.563,5,7,5,2,41
133,2026-01-10T00:00:00,MIN @ CLE,L,12,20.463333,5,8,0.625,2,2,7,2,-5
206,2025-12-29T00:00:00,MIN @ CHI,W,12,14.366667,5,8,0.625,1,4,1,0,12
258,2025-12-19T00:00:00,MIN vs. OKC,W,13,15.678333,4,7,0.571,4,6,3,3,-1
269,2025-12-17T00:00:00,MIN vs. MEM,L,12,34.138333,3,11,0.273,2,9,5,3,10
277,2025-12-14T00:00:00,MIN vs. SAC,W,18,35.683333,6,10,0.600,1,4,5,3,10
294,2025-12-08T00:00:00,MIN vs. PHX,L,14,15.778333,5,8,0.625,4,6,3,1,7
457,2025-11-07T00:00:00,MIN vs. UTA,W,12,8.733333,4,4,1.000,2,2,3,1,4
631,2025-10-04T00:00:00,MIN @ DEN,W,18,22.083333,5,8,0.625,3,4,3,3,5



Comparison:
Total games played by Bones Hyland: 48
Overall record when Bones Hyland plays: 28-20 (0.583 / 58.3%)
